## Use Case Overview: 

The objective of this notebook is to predict the duration of NYC taxi trips. \
We use data related to yellow or green taxis to train a simple prediction model.

In this notebook, we use:

- Yellow taxi data from January 2021 for model training.
- Data from February 2021 to test the model (make predictions).

**Features for Model Training**: Useful variables related to the trip itself 

**Target to Predict**: The duration of the trip


### Data
Data for the whole course can be downloaded following this [link](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page) or using the following code:



Run `uv sync` from the repository root and select the project's Python kernel before running all cells. Data downloads are cached locally and are not committed.

In [ ]:
from pathlib import Path

from urllib.request import urlretrieve

REPO_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "pyproject.toml").is_file())
DATA_FOLDER = REPO_ROOT / "data"
train_path = f"{DATA_FOLDER}/yellow_tripdata_2021-01.parquet"
test_path = f"{DATA_FOLDER}/yellow_tripdata_2021-02.parquet"
predict_path = f"{DATA_FOLDER}/yellow_tripdata_2021-03.parquet"


DATA_FOLDER.mkdir(parents=True, exist_ok=True)
for file_path in (train_path, test_path, predict_path):
    destination = Path(file_path)
    if not destination.is_file():
        url = f"https://d37ci6vzurychx.cloudfront.net/trip-data/{destination.name}"
        temporary_path = destination.with_suffix(".parquet.part")
        urlretrieve(url, temporary_path)
        temporary_path.replace(destination)
    print(f"Ready: {destination.name}")

In [ ]:
from pathlib import Path
from typing import Dict, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_percentage_error, root_mean_squared_error

**Info** : <p style=color:green>Using large amounts of data in jupyter notebook, some cell can take some time to run<p/>

# 1 - Load data

In [ ]:
train_df = pd.read_parquet(Path(DATA_FOLDER) / "yellow_tripdata_2021-01.parquet")

In [ ]:
train_df.head()

In [ ]:
train_df.info()

In [ ]:
train_df.isnull().sum()

# 2 - Prepare the data

## 2-1 Compute the target

The dataset contains pickup and dropoff time but not the duration itself. \
We compute the duration of a taxi trip in minutes using these two variables.

In [ ]:
def compute_target(df: pd.DataFrame) -> pd.DataFrame:
    """Compute trip duration in minutes without modifying the input."""
    df = df.copy()
    df["duration"] = df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]
    df["duration"] = df["duration"].dt.total_seconds() / 60
    return df


train_df = compute_target(train_df)

We can visualise how the duration is distributed : 

In [ ]:
train_df["duration"].describe()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
sns.histplot(train_df["duration"], bins=100, ax=ax)
ax.set(title="Trip durations before filtering", xlabel="Duration (minutes)", yscale="log")
plt.show()

We notice that there are negative durations and trips that last 6 hours. \
We will proceed to remove outliers and narrow the scope to trips lasting between 1 minute and 1 hour.

In [ ]:
MIN_DURATION = 1
MAX_DURATION = 60


def filter_outliers(df: pd.DataFrame, min_duration: int = MIN_DURATION, max_duration: int = MAX_DURATION) -> pd.DataFrame:
    """Keep trips within the inclusive duration limits."""
    df = df.loc[df["duration"].between(min_duration, max_duration)].copy()
    return df


train_df = filter_outliers(train_df)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
sns.histplot(train_df["duration"], bins=60, ax=ax)
ax.set(title="Trip durations after filtering", xlabel="Duration (minutes)")
plt.show()

## 2-2 Prepare features

### 2-2-1 Categorical features

The inputs are pickup zone, dropoff zone and passenger count, all available when booking a trip. Zone IDs are labels, not continuous measurements. We encode them as strings so `DictVectorizer` one-hot encodes them; pandas `category` alone would leave integer values in the dictionaries. Missing values use the `-1` category. The target is trip duration in minutes, computed from the timestamps. Dropoff time is only used for the target, never as an input (which would leak the answer).

In [ ]:
CATEGORICAL_COLS = ["PULocationID", "DOLocationID", "passenger_count"]


def encode_categorical_cols(df: pd.DataFrame) -> pd.DataFrame:
    """Encode categorical IDs as strings for DictVectorizer."""
    df = df.copy()
    df[CATEGORICAL_COLS] = df[CATEGORICAL_COLS].fillna(-1).astype("int").astype(str)
    return df


train_df = encode_categorical_cols(train_df)

In [ ]:
def extract_x_y(df: pd.DataFrame, dv: DictVectorizer = None) -> Tuple[np.array, np.array, DictVectorizer]:
    """Extract sparse features and target, fitting only on training data."""
    dicts = df[CATEGORICAL_COLS].to_dict(orient="records")
    if dv is None:
        dv = DictVectorizer()
        dv.fit(dicts)
    X = dv.transform(dicts)
    y = df["duration"].values
    return X, y, dv


X_train, y_train, dv = extract_x_y(train_df)
assert X_train.shape[1] > len(CATEGORICAL_COLS), "Expected one-hot encoded features"
assert np.all(X_train.getnnz(axis=1) == len(CATEGORICAL_COLS))
print(f"Training trips: {X_train.shape[0]:,}; one-hot features: {X_train.shape[1]}")

# 3 - Train model

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)

In [ ]:
# ~ 2.5 minutes
rf = RandomForestRegressor(n_estimators=100, max_depth=11, max_features="sqrt", random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)

# 4 - Evaluate model

In [ ]:
def compute_metrics(y: np.array, y_pred: np.array) -> Dict[str, float]:
    """Compute RMSE and MAPE."""
    metrics = {
        "rmse": root_mean_squared_error(y, y_pred),
        "mape": mean_absolute_percentage_error(y, y_pred),
    }
    return metrics

## 4-1 On train data

In [ ]:
y_pred_lr = lr.predict(X_train)
y_pred_rf = rf.predict(X_train)
compute_metrics(y_train, y_pred_lr), compute_metrics(y_train, y_pred_rf)

## 4-2 On test data

In [ ]:
test_df = pd.read_parquet(Path(DATA_FOLDER) / "yellow_tripdata_2021-02.parquet")

In [ ]:
test_df = compute_target(test_df)
test_df = filter_outliers(test_df)
test_df = encode_categorical_cols(test_df)
X_test, y_test, _ = extract_x_y(test_df, dv=dv)

In [ ]:
y_pred_test_lr = lr.predict(X_test)
y_pred_test_rf = rf.predict(X_test)
assert X_test.shape[1] == X_train.shape[1]
assert np.isfinite(y_pred_test_lr).all() and np.isfinite(y_pred_test_rf).all()
results = pd.DataFrame(
    [
        {"model": name, "split": split, **compute_metrics(y, predictions)}
        for name, split, y, predictions in [
            ("Linear regression", "train", y_train, y_pred_lr),
            ("Linear regression", "test", y_test, y_pred_test_lr),
            ("Random forest", "train", y_train, y_pred_rf),
            ("Random forest", "test", y_test, y_pred_test_rf),
            ("Training mean baseline", "test", y_test, np.full(y_test.shape, y_train.mean())),
        ]
    ]
)
results.round(4)

# Conclusion

* What can you say of the trained models?
* If you had to improve these models, what would you do? 

### What can you say of the trained models?

Both models use one-hot encoded pickup zone, dropoff zone and passenger count to predict trip duration. Linear regression learns additive effects; the random forest can learn nonlinear interactions, but its depth is limited to 11. Training uses January 2021 and evaluation uses February 2021, with the vectorizer fitted on January only.

Results from running the full datasets (durations restricted to 1–60 minutes):

| Model | Train RMSE (min) | Test RMSE (min) | Train MAPE | Test MAPE |
| --- | ---: | ---: | ---: | ---: |
| Linear regression | 6.7824 | 7.3711 | 67.65% | 63.45% |
| Random forest | 7.7976 | 8.3327 | 80.19% | 74.55% |
| Training-mean baseline | — | 9.2953 | — | 81.13% |

Linear regression is the better model here: it has lower training and test errors than this random forest, and reduces test RMSE by about 21% relative to always predicting the training mean. More complexity does not automatically improve predictions. The forest's worse training performance suggests underfitting with the current depth and feature-sampling settings on sparse one-hot features, rather than strong overfitting. Both models have somewhat higher test RMSE, but lower test MAPE; different monthly trip distributions and metric weighting can explain this, so the gap alone does not prove overfitting.

RMSE is measured in minutes and penalizes large errors; it is not the mean absolute error. MAPE is returned as a fraction and is shown above as a percentage. It heavily weights short trips, so the high relative errors should be interpreted alongside absolute errors. These results apply only to trips lasting 1–60 minutes, not all taxi trips.

### If you had to improve these models, what would you do?

- Add features available at booking time: pickup hour, day of week, holidays, pickup/dropoff zone pairs, estimated route distance, and weather or traffic forecasts. Do not use actual dropoff time, final fare or recorded trip distance, which are only known after the trip.
- Tune regularization for linear regression and the forest's depth, number of trees, minimum leaf size and feature sampling. Try gradient boosting with suitable categorical handling.
- Use chronological validation or rolling monthly splits for model selection, keeping a later month untouched for final testing. Do not repeatedly tune against February and still call it an unbiased test.
- Inspect residuals by route, pickup hour and trip duration; add MAE and error percentiles to quantify typical errors and bad cases. Investigate missing passenger counts and rare or unseen zones.
- Check performance on newer months and monitor distribution drift. Revisit the duration filter if the intended service also needs to support very short or long trips.